# Imports

In [ ]:
import os
import gc
import time
import pickle
import numpy as np
import pandas as pd

from PIL import Image
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from torchvision import models, transforms

from transformers import (
    ViTModel,
    AutoModel,
    AutoImageProcessor,
    AutoModelForImageClassification
)

from captum.attr import (
    IntegratedGradients,
    Saliency,
    GradientShap,
    Occlusion
)

from quantus.metrics.faithfulness.faithfulness_correlation import (
    FaithfulnessCorrelation
)

from quantus.metrics.complexity.complexity import Complexity
from quantus.metrics.complexity.sparseness import Sparseness

from scipy.stats import spearmanr

# Configuration

In [8]:
CURRENT_DATASET = "Flickr8k"

BASE_DIR = "TFE_Data"
DATASETS_DIR = os.path.join(BASE_DIR, "Datasets")

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print("Using device:", device)

Using device: cuda


In [9]:
from paths import EMBED_DIR

# Dataset Loader

In [ ]:
class ImageDataset(torch.utils.data.Dataset):

    def __init__(self, image_paths, transform=None):
        self.image_paths = image_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):

        path = self.image_paths[idx]
        try:
            img = Image.open(path).convert("RGB")

        except:
            img = Image.new("RGB", (224, 224), (0, 0, 0))

        if self.transform:
            img = self.transform(img)

        return img

# Vision Models

## ResNet50

In [11]:
def get_resnet50_model(device):
    weights = models.ResNet50_Weights.DEFAULT
    model = models.resnet50(weights=weights)
    model.fc = nn.Linear(2048, 1000)
    return model.to(device).eval(), weights.transforms()


## MobileNetV3

In [12]:
def get_mobilenet_v3_model(device):
    weights = models.MobileNet_V3_Large_Weights.DEFAULT
    model = models.mobilenet_v3_large(weights=weights)
    model.classifier[3] = nn.Linear(1280, 1000)
    return model.to(device).eval(), weights.transforms()


## ViT

In [13]:
class ViTWithHead(nn.Module):
    def __init__(self, device):
        super().__init__()
        self.backbone = ViTModel.from_pretrained("google/vit-base-patch16-224-in21k")
        self.head = nn.Linear(self.backbone.config.hidden_size, 1000)

    def forward(self, x):
        out = self.backbone(pixel_values=x)
        cls = out.last_hidden_state[:, 0]
        return self.head(cls)

def get_vit_model(device):
    transform = transforms.Compose([
        transforms.Resize((224,224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ])
    return ViTWithHead(device).to(device).eval(), transform


## PVT

In [14]:
class CaptumWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        outputs = self.model(pixel_values=x)
        return outputs.logits


In [15]:
class PVTWithHead(nn.Module):
    def __init__(self, device):
        super().__init__()
        self.backbone = AutoModel.from_pretrained("Zetatech/pvt-tiny-224").to(device).eval()

        hidden = self.backbone.config.embed_dims[-1]

        self.head = nn.Linear(hidden, 1000).to(device)

    def forward(self, x):
        out = self.backbone(pixel_values=x)
        cls = out.last_hidden_state[:, 0, :]   # CLS token
        return self.head(cls)

from transformers import AutoModelForImageClassification, AutoImageProcessor

def get_pvt_model(device):
    processor = AutoImageProcessor.from_pretrained(
        "Zetatech/pvt-tiny-224"
    )

    base_model = AutoModelForImageClassification.from_pretrained(
        "Zetatech/pvt-tiny-224"
    )

    model = CaptumWrapper(base_model).to(device).eval()

    def transform(img):

        return processor(
            images=img,
            return_tensors="pt"
        )["pixel_values"].squeeze(0)

    return model, transform



# Captum

In [16]:
def explain_ig(model, img, label):
    ig = IntegratedGradients(model)
    return ig.attribute(img.unsqueeze(0), target=label)

def explain_saliency(model, img, label):
    sal = Saliency(model)
    return sal.attribute(img.unsqueeze(0), target=label)

def explain_gradientShap(model, img, label):
    gs = GradientShap(model)
    baseline = torch.zeros_like(img)
    return gs.attribute(img.unsqueeze(0), baselines=baseline.unsqueeze(0), target=label)

def explain_occlusion(model, img, label):
    occ = Occlusion(model)
    return occ.attribute(
        img.unsqueeze(0),
        target=label,
        sliding_window_shapes=(3,15,15),
        strides=(3,8,8)
    )


# Quantus

In [17]:
def reduce_attr_for_quantus(attr):
    # attr may be (C,H,W), (1,C,H,W), (B,C,H,W), (B,1,C,H,W)
    if attr.dim() == 3:
        attr = attr.unsqueeze(0)  # (1,C,H,W)

    if attr.dim() == 5:
        attr = attr.squeeze(1)  # (B,C,H,W)

    if attr.dim() != 4:
        raise ValueError(f"Expected (B,C,H,W), got {attr.shape}")

    a = F.interpolate(attr, size=(224,224), mode="bilinear", align_corners=False)

    return a.detach().cpu().numpy()


In [18]:
def evaluate_quantus(model, img, label, attr):
    model_cpu = model.to("cpu")
    model_cpu.eval()

    x_np = img.unsqueeze(0).detach().cpu().numpy()
    a_np = reduce_attr_for_quantus(attr)
    #a_np = np.expand_dims(a_np, axis=0)
    y_np = np.array([label])

    metrics = [
        FaithfulnessCorrelation(),
        Complexity(),
        Sparseness()
    ]

    results = {}
    for m in metrics:
        try:
            results[m.__class__.__name__] = m(
                model=model_cpu,
                x_batch=x_np,
                a_batch=a_np,
                y_batch=y_np
            )
        except Exception as e:
            print(f"[Quantus error] {m.__class__.__name__}: {e}")
            results[m.__class__.__name__] = np.nan

    model.to(device)
    return results


In [19]:
from quantus.metrics.robustness.local_lipschitz_estimate import LocalLipschitzEstimate

def evaluate_quantus_robustness(model, img, label):
    model_q = model.to("cpu").eval()

    x_np = img.detach().cpu().numpy()
    y_np = np.array([label])

    metric = LocalLipschitzEstimate()

    try:
        score = metric(
            model=model_q,
            x_batch=x_np,
            y_batch=y_np
        )
    except Exception:
        score = np.nan

    return score


In [20]:
def metric_rank_correlation(model, img, label, attr, patch=16):
    model_cpu = model.to("cpu").eval()
    img_cpu = img.to("cpu")

    # Attribution IG → numpy
    a = attr.detach().cpu().numpy().squeeze()
    a = np.mean(a, axis=0)  # HxW

    H, W = a.shape
    impacts = []
    attrs_patched = []

    with torch.no_grad():
        base = model_cpu(img_cpu).softmax(dim=1)[0, label].item()

        for i in range(0, H, patch):
            for j in range(0, W, patch):

                # 1) Impact du patch
                img_mod = img_cpu.clone()
                img_mod[:, :, i:i+patch, j:j+patch] = 0
                out = model_cpu(img_mod).softmax(dim=1)[0, label].item()
                impacts.append(base - out)

                # 2) Attribution moyenne dans ce patch
                patch_attr = a[i:i+patch, j:j+patch]
                attrs_patched.append(patch_attr.mean())

    impacts = np.array(impacts)
    attrs_patched = np.array(attrs_patched)

    corr, _ = spearmanr(attrs_patched, impacts)
    return float(corr)


In [21]:
def quantus_explain_func(model, inputs, targets, **kwargs):
    model.eval()
    ig = IntegratedGradients(model)

    attributions = []
    for i in range(inputs.shape[0]):
        attr = ig.attribute(
            inputs[i].unsqueeze(0),
            target=int(targets[i])
        )
        attributions.append(attr)

    return torch.cat(attributions, dim=0).detach().cpu().numpy()

In [22]:
def evaluate_quantus_static(model, img, label, attr):
    model_q = model.to("cpu").eval()

    x_np = img.detach().cpu().numpy()
    a_np = reduce_attr_for_quantus(attr)

    a_np = np.mean(a_np, axis=1, keepdims=True)

    y_np = np.array([label])

    metrics = [
        FaithfulnessCorrelation(),
        Sparseness(),
        Complexity()
    ]

    results = {}
    for m in metrics:
        try:
            results[m.__class__.__name__] = m(
                model=model_q,
                x_batch=x_np,
                a_batch=a_np,
                y_batch=y_np
            )
        except Exception as e:
            results[m.__class__.__name__] = np.nan

    return results

# Execution

In [23]:
ATTR_DIR = "saved_attributions"
os.makedirs(ATTR_DIR, exist_ok=True)

all_attrs = []

df = pd.read_pickle(os.path.join(DATASETS_DIR, f"df_{CURRENT_DATASET}.pkl"))
IMAGE_PATHS = df["image_path"].tolist()

MAX_IMAGES = 50
IMAGE_PATHS_50 = IMAGE_PATHS[:50]
IMAGE_PATHS_ALL = IMAGE_PATHS

In [24]:
models = {
    "ResNet50": get_resnet50_model(device),
    "MobileNetV3": get_mobilenet_v3_model(device),  
    "ViT": get_vit_model(device),
    "PVT": get_pvt_model(device),
}

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/177 [00:00<?, ?it/s]

In [ ]:
# 50 image subset for testing
for name, (model, transform) in models.items():
    print(f"\n=== Running {name} (Captum only) ===")

    dataset = ImageDataset(IMAGE_PATHS_50, transform)
    loader = torch.utils.data.DataLoader(dataset, batch_size=1)

    model_attrs = []

    for idx, img in enumerate(loader):
        img = img.squeeze(0).to(device)

        logits = model(img.unsqueeze(0))
        label = logits.argmax(dim=1).item()

        ig_attr  = explain_ig(model, img, label)
        sal_attr = explain_saliency(model, img, label)
        gs_attr  = explain_gradientShap(model, img, label)
        occ_attr = explain_occlusion(model, img, label)

        model_attrs.append({
            "image_idx": idx,
            "label": label,
            "IG": ig_attr.cpu(),
            "Saliency": sal_attr.cpu(),
            "GradientShap": gs_attr.cpu(),
            "Occlusion": occ_attr.cpu(),
        })

    all_attrs.append({"model": name, "attrs": model_attrs})

with open("captum_attributions.pkl", "wb") as f:
    pickle.dump(all_attrs, f)


In [ ]:
os.makedirs("captum_ig", exist_ok=True)

for name, (model, transform) in models.items():
    print(f"\n=== Running {name} (IG only) ===")

    dataset = ImageDataset(IMAGE_PATHS_ALL, transform)
    loader = torch.utils.data.DataLoader(dataset, batch_size=1)

    model_dir = f"captum_ig/{name}"
    os.makedirs(model_dir, exist_ok=True)

    for idx, img in enumerate(loader):

        out_path = f"{model_dir}/{idx}.pkl"
        if os.path.exists(out_path):
            continue  # already computed

        img = img.squeeze(0).to(device)

        logits = model(img.unsqueeze(0))
        label = logits.argmax(dim=1).item()

        ig_attr = explain_ig(model, img, label)

        with open(out_path, "wb") as f:
            pickle.dump({
                "image_idx": idx,
                "label": label,
                "IG": ig_attr.cpu(),
            }, f)

        print(f"{name}: saved image {idx}")


In [ ]:
def to_numpy(attr):
    return attr.detach().cpu().numpy().squeeze()
def normalize(a):
    a = np.abs(a)
    return (a - a.min()) / (a.max() - a.min() + 1e-8)
from scipy.stats import spearmanr
import numpy as np

def flatten(attr):
    return normalize(to_numpy(attr)).flatten()
def compute_agreements(sample):
    
    ig = flatten(sample["IG"])
    sal = flatten(sample["Saliency"])
    gs  = flatten(sample["GradientShap"])
    occ = flatten(sample["Occlusion"])

    return {
        "ig_sal": spearmanr(ig, sal).correlation,
        "ig_gs": spearmanr(ig, gs).correlation,
        "ig_occ": spearmanr(ig, occ).correlation,
        "sal_occ": spearmanr(sal, occ).correlation,
        "gs_occ": spearmanr(gs, occ).correlation,
    }
agreement_results = []

for model_entry in all_attrs:
    model_name = model_entry["model"]

    for sample in model_entry["attrs"]:
        res = compute_agreements(sample)
        res["model"] = model_name
        res["image_idx"] = sample["image_idx"]
        agreement_results.append(res)

df_agree = pd.DataFrame(agreement_results)

In [ ]:
df_agree

In [ ]:
def occlusion_proxy_corr(sample):
    
    ig = normalize(to_numpy(sample["IG"]))
    occ = normalize(to_numpy(sample["Occlusion"]))

    return spearmanr(ig.flatten(), occ.flatten()).correlation
df_agree["ig_occ_corr"] = df_agree.apply(
    lambda row: row["ig_occ"], axis=1
)

df_agree.groupby("model").mean(numeric_only=True)

model_summary = df_agree.groupby("model")[[
    "ig_sal",
    "ig_gs",
    "ig_occ",
    "sal_occ",
    "gs_occ"
]].mean().reset_index()

model_summary

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))

sns.heatmap(
    model_summary.set_index("model"),
    annot=True,
    cmap="viridis",
    vmin=-1,
    vmax=1
)

plt.title("Cross-Method Attribution Agreement per Model")
plt.show()

model_summary["explanation_quality"] = (
    model_summary["ig_occ"] * 0.5 +
    model_summary["ig_gs"] * 0.3 +
    model_summary["ig_sal"] * 0.2
)

model_summary.sort_values("explanation_quality", ascending=False)

In [ ]:
def mask_topk(img, attr, k=0.1):
    
    a = normalize(to_numpy(attr))
    threshold = np.quantile(a, 1 - k)

    mask = a < threshold
    mask = torch.tensor(mask).to(img.device)

    img_mod = img.clone()
    img_mod[:, mask] = 0

    return img_mod
def faithfulness_test(model, img, attr, label):

    with torch.no_grad():
        base = model(img.unsqueeze(0)).softmax(1)[0, label]

        masked = mask_topk(img, attr)
        new = model(masked.unsqueeze(0)).softmax(1)[0, label]

    return (base - new).item()
def test_all_methods(model, img, sample, label):

    return {
        "IG": faithfulness_test(model, img, sample["IG"], label),
        "Saliency": faithfulness_test(model, img, sample["Saliency"], label),
        "GradientShap": faithfulness_test(model, img, sample["GradientShap"], label),
    }
    

all_faithfulness = []    
for model_entry in all_attrs:
    model_name = model_entry["model"]

    for sample in model_entry["attrs"]:
        img_idx = sample["image_idx"]
        label = sample["label"]

        faithfulness_scores = test_all_methods(
            models[model_name][0], img, sample, label
        )

        print(f"Model: {model_name}, Image idx: {img_idx}, Scores: {faithfulness_scores}")
        all_faithfulness.append(f"Model: {model_name}, Image idx: {img_idx}, Scores: {faithfulness_scores}")

In [ ]:
quantus_results = []

for entry in all_attrs:
    model_name = entry["model"]
    print(f"\n=== Quantus for {model_name} ===")

    model, transform = models[model_name]
    model.eval()

    model_scores = []

    for item in entry["attrs"]:
        idx = item["image_idx"]
        label = item["label"]

        img = transform(Image.open(IMAGE_PATHS[idx])).to(device)
        img = img.unsqueeze(0)

        attr = item["IG"].to(device)

        scores = evaluate_quantus(model, img, label, attr)

        model_scores.append(scores)

    quantus_results.append({
        "model": model_name,
        "scores": model_scores
    })


In [ ]:
for entry in quantus_results:
    model_name = entry["model"]
    print(f"\n=== Adding Robustness for {model_name} ===")

    model, transform = models[model_name]
    model.eval()

    for i, item in enumerate(entry["scores"]):
        idx = all_attrs[[m["model"] for m in all_attrs].index(model_name)]["attrs"][i]["image_idx"]
        label = all_attrs[[m["model"] for m in all_attrs].index(model_name)]["attrs"][i]["label"]

        # Load image
        img = transform(Image.open(IMAGE_PATHS[idx])).to(device)
        img = img.unsqueeze(0)

        # Compute robustness
        robustness_score = evaluate_quantus_robustness(model, img, label)

        # Add to existing scores
        entry["scores"][i]["Robustness"] = robustness_score


In [ ]:
def metric_entropy(attr):
    a = attr.detach().cpu().numpy().squeeze()
    a = np.mean(a, axis=0)
    a = np.abs(a)
    a = a / (a.sum() + 1e-8)
    entropy = -np.sum(a * np.log(a + 1e-8))
    return entropy


In [ ]:
import torchvision.transforms as T

augmentations = [
    T.RandomRotation(5),
    T.ColorJitter(brightness=0.1, contrast=0.1),
    T.RandomHorizontalFlip(p=1.0)
]

def metric_stability(model, img, label, attr_original):
    ig = IntegratedGradients(model)
    sims = []

    for aug in augmentations:
        img_aug = aug(img.squeeze(0)).unsqueeze(0)

        attr_aug = ig.attribute(img_aug, target=label)
        a1 = attr_original.detach().cpu().numpy().flatten()
        a2 = attr_aug.detach().cpu().numpy().flatten()

        # similarité cosinus
        sim = np.dot(a1, a2) / (np.linalg.norm(a1)*np.linalg.norm(a2) + 1e-8)
        sims.append(sim)

    return float(np.mean(sims))


In [ ]:
for entry in quantus_results:
    model_name = entry["model"]
    print(f"\n=== Adding new XAI metrics for {model_name} ===")

    model, transform = models[model_name]
    model.eval()

    for i, score_dict in enumerate(entry["scores"]):
        idx = all_attrs[[m["model"] for m in all_attrs].index(model_name)]["attrs"][i]["image_idx"]
        label = all_attrs[[m["model"] for m in all_attrs].index(model_name)]["attrs"][i]["label"]
        attr = all_attrs[[m["model"] for m in all_attrs].index(model_name)]["attrs"][i]["IG"]

        img = transform(Image.open(IMAGE_PATHS[idx])).to(device)
        img = img.unsqueeze(0)

        # Rank correlation
        score_dict["RankCorrelation"] = metric_rank_correlation(model, img, label, attr)

        # Entropy
        score_dict["Entropy"] = metric_entropy(attr)


In [ ]:
def metric_stability(model, img, label, attr_original):
    model_cpu = model.to("cpu").eval()
    img_cpu = img.to("cpu")

    ig = IntegratedGradients(model_cpu)

    augmentations = [
        T.RandomRotation(5),
        T.ColorJitter(brightness=0.1, contrast=0.1),
        T.RandomHorizontalFlip(p=1.0)
    ]

    sims = []

    for aug in augmentations:
        img_aug = aug(img_cpu.squeeze(0)).unsqueeze(0)

        attr_aug = ig.attribute(img_aug, target=label)

        a1 = attr_original.detach().cpu().numpy().flatten()
        a2 = attr_aug.detach().cpu().numpy().flatten()

        sim = np.dot(a1, a2) / (np.linalg.norm(a1)*np.linalg.norm(a2) + 1e-8)
        sims.append(sim)

    return float(np.mean(sims))


In [ ]:
for entry in quantus_results:
    model_name = entry["model"]
    print(f"\n=== Adding new XAI metrics for {model_name} ===")

    model, transform = models[model_name]
    model.eval()

    for i, score_dict in enumerate(entry["scores"]):
        idx = all_attrs[[m["model"] for m in all_attrs].index(model_name)]["attrs"][i]["image_idx"]
        label = all_attrs[[m["model"] for m in all_attrs].index(model_name)]["attrs"][i]["label"]
        attr = all_attrs[[m["model"] for m in all_attrs].index(model_name)]["attrs"][i]["IG"]

        img = transform(Image.open(IMAGE_PATHS[idx])).to(device)
        img = img.unsqueeze(0)

        # Rank correlation
        score_dict["Stability"] = metric_stability(model, img, label, attr)


In [ ]:
rows = []

for entry in quantus_results:
    model = entry["model"]
    for s in entry["scores"]:
        for metric, value in s.items():

            # Convert lists or arrays to a single float
            if isinstance(value, (list, np.ndarray)):
                if len(value) == 0:
                    val = np.nan
                else:
                    val = float(np.mean(value))
            else:
                try:
                    val = float(value)
                except:
                    val = np.nan

            rows.append([model, metric, val])


df = pd.DataFrame(rows, columns=["model", "metric", "value"])
summary = df.groupby(["model", "metric"]).mean().reset_index()


summary

In [26]:
# ------------------------------------------------------------
# Fix model names BEFORE saving the CSV
# ------------------------------------------------------------
df_vision_wide = pd.read_csv("/home/aysel/tfe/Explainability_Vision.csv")

df_vision_wide["model"] = (
    df_vision_wide["model"]
    .str.strip()
    .str.lower()
    .str.replace("-", "_", regex=False)
    .str.replace("mobilenetv3", "mobilenet_v3", regex=False)
    .str.replace("clip_vision", "clip_vision", regex=False)
    .str.replace("resnet50", "resnet50", regex=False)
    .str.replace("vit", "vit", regex=False)
    .str.replace("pvt", "pvt", regex=False)
)
df_vision_wide.to_csv("Explainability_Vision.csv", index=False)

df_vision_wide

,model,faithfulness,complexity,sparsity,rank_corr
0,clip_vision,0.008913,10.084119,0.624340,0.062989
1,mobilenet_v3,-0.022481,10.129820,0.605527,0.092867
2,pvt,-0.041493,9.989751,0.643774,0.105736
3,resnet50,-0.016260,10.103036,0.614844,0.084531
4,vit,0.008630,10.132332,0.600143,0.095724


In [ ]:
import pickle
import os

def load_all_ig():
    all_attrs = []

    for model_name in models.keys():
        model_dir = f"captum_ig/{model_name}"
        model_attrs = []

        for file in sorted(os.listdir(model_dir)):
            with open(os.path.join(model_dir, file), "rb") as f:
                model_attrs.append(pickle.load(f))

        all_attrs.append({"model": model_name, "attrs": model_attrs})

    return all_attrs

all_attrs = load_all_ig()


In [ ]:
quantus_results = []

for entry in all_attrs:
    model_name = entry["model"]
    print(f"\n=== Quantus for {model_name} ===")

    model, transform = models[model_name]
    model.eval()

    model_scores = []

    for item in entry["attrs"]:
        idx = item["image_idx"]
        label = item["label"]

        # Load image
        img = transform(Image.open(IMAGE_PATHS[idx])).to(device)
        img = img.unsqueeze(0)  # (1, C, H, W)

        # Load IG attribution
        attr = item["IG"].to(device)  # (1, C, H, W)

        # Run Quantus
        scores = evaluate_quantus(model, img, label, attr)

        model_scores.append(scores)


    quantus_results.append({
        "model": model_name,
        "scores": model_scores
    })


In [ ]:
for entry in quantus_results:
    model_name = entry["model"]
    print(f"\n=== Adding Robustness for {model_name} ===")

    model, transform = models[model_name]
    model.eval()

    for i, score_dict in enumerate(entry["scores"]):
        item = all_attrs[[m["model"] for m in all_attrs].index(model_name)]["attrs"][i]

        idx = item["image_idx"]
        label = item["label"]

        img = transform(Image.open(IMAGE_PATHS[idx])).to(device)
        img = img.unsqueeze(0)

        robustness_score = evaluate_quantus_robustness(model, img, label)
        score_dict["Robustness"] = robustness_score


In [ ]:
for entry in quantus_results:
    model_name = entry["model"]
    print(f"\n=== Adding XAI metrics for {model_name} ===")

    model, transform = models[model_name]
    model.eval()

    for i, score_dict in enumerate(entry["scores"]):
        item = all_attrs[[m["model"] for m in all_attrs].index(model_name)]["attrs"][i]

        idx = item["image_idx"]
        label = item["label"]
        attr = item["IG"]

        img = transform(Image.open(IMAGE_PATHS[idx])).to(device)
        img = img.unsqueeze(0)

        score_dict["RankCorrelation"] = metric_rank_correlation(model, img, label, attr)
        score_dict["Entropy"] = metric_entropy(attr)
        score_dict["Stability"] = metric_stability(model, img, label, attr)


In [ ]:
rows = []

for entry in quantus_results:
    model = entry["model"]
    for s in entry["scores"]:
        for metric, value in s.items():

            if isinstance(value, (list, np.ndarray)):
                val = float(np.mean(value)) if len(value) > 0 else np.nan
            else:
                try:
                    val = float(value)
                except:
                    val = np.nan

            rows.append([model, metric, val])

df = pd.DataFrame(rows, columns=["model", "metric", "value"])
summary = df.groupby(["model", "metric"]).mean().reset_index()
